In [1]:
from music21 import converter

def analyze_midi(file):
    midi = converter.parse(file)

    notes = list(midi.flatten().notes)

    duration = midi.duration.quarterLength
    note_count = len(notes)

    density = note_count / duration if duration > 0 else 0

    tempos = midi.metronomeMarkBoundaries()
    bpm = tempos[0][2].number if tempos else "Unknown"

    print("Tempo (BPM):", bpm)
    print("Total Notes:", note_count)
    print("Duration:", duration)
    print("Note Density:", round(density, 2))


In [2]:
analyze_midi("generated_Classical.mid")

Tempo (BPM): 90
Total Notes: 400
Duration: 400.0
Note Density: 1.0


In [3]:
from music21 import converter
import numpy as np


In [4]:
def extract_features(midi_file):
    midi = converter.parse(midi_file)
    notes = list(midi.flatten().notes)

    # --- Tempo ---
    tempos = midi.metronomeMarkBoundaries()
    bpm = tempos[0][2].number if tempos else 100

    # --- Duration ---
    duration = midi.duration.quarterLength

    # --- Note Density ---
    density = len(notes) / duration if duration > 0 else 0

    # --- Pitch Information ---
    pitches = []
    for n in notes:
        if n.isNote:
            pitches.append(n.pitch.midi)
        elif n.isChord:
            pitches.extend(p.midi for p in n.pitches)

    pitch_range = max(pitches) - min(pitches) if pitches else 0
    pitch_std = np.std(pitches) if pitches else 0

    return {
        "bpm": bpm,
        "density": density,
        "pitch_range": pitch_range,
        "pitch_std": pitch_std
    }


In [5]:
GENRE_RULES = {
    "Classical": {"bpm": (60, 100), "density": (2, 6), "pitch_std": (10, 25)},
    "Jazz":      {"bpm": (90, 130), "density": (4, 9), "pitch_std": (15, 30)},
    "Rock":      {"bpm": (110, 160), "density": (6, 14), "pitch_std": (8, 20)},
    "EDM":       {"bpm": (120, 135), "density": (8, 16), "pitch_std": (5, 15)},
    "Hip-Hop":   {"bpm": (70, 100), "density": (4, 8), "pitch_std": (6, 18)},
    "Ambient":   {"bpm": (40, 80), "density": (1, 5), "pitch_std": (12, 28)}
}


In [8]:
def calculate_accuracy(features, expected_genre):

    score = 0

    if expected_genre == "Rock":

        if 120 <= features['bpm'] <= 180:
            score += 25

        if 0.6 <= features['density'] <= 1.5:
            score += 25

        if 40 <= features['pitch_range'] <= 80:
            score += 25

        if 8 <= features['pitch_std'] <= 20:
            score += 25

    return score

In [9]:
file = "generated_Rock.mid"
expected_genre = "Rock"

features = extract_features(file)

print("Extracted Features:", features)

accuracy = calculate_accuracy(features, expected_genre)
print(f"Genre Accuracy: {accuracy:.2f}/100")


Extracted Features: {'bpm': 140, 'density': 1.0, 'pitch_range': 68, 'pitch_std': 12.67015496265811}
Genre Accuracy: 100.00/100
